In [ ]:
import kagglehub

handle = "uditmaurya1588/models/pytorch/resnet"
model_dir = kagglehub.model_download(handle)

print("Downloaded model directory:", model_dir)

In [ ]:
def load_audio(path, sr=22050, duration=5):
    audio, _ = librosa.load(path, sr=sr)
    max_len = sr * duration
    if len(audio) < max_len:
        audio = np.pad(audio, (0, max_len - len(audio)))
    return audio[:max_len]




def audio_to_mel(audio, sr=22050):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=128,
        n_fft=2048,
        hop_length=512
    )
    mel = librosa.power_to_db(mel)

    # Normalize per sample
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    return mel.astype(np.float32)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GenreResNet(num_classes=10).to(device)

model_file = "/kaggle/input/models/uditmaurya1588/models/pytorch/resnet/1/resnet.pth"

state_dict = torch.load(model_file, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print("Model loaded successfully")

In [ ]:
class MashupTestDataset(Dataset):
    def __init__(self, test_csv, mashup_dir):
        self.df = pd.read_csv(test_csv)
        self.mashup_dir = mashup_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Adjust column name if needed
        filename = row["filename"] if "filename" in row else row["id"] + ".wav"

        audio_path = os.path.join(self.mashup_dir, filename)

        audio = load_audio(audio_path)
        mel = audio_to_mel(audio)

        mel = torch.tensor(mel).unsqueeze(0)

        return mel, row["id"]

In [ ]:
test_csv = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
mashup_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

test_dataset = MashupTestDataset(test_csv, mashup_dir)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0   # Important for Kaggle
)

In [ ]:
genres = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]

predictions = []

model.eval()

with torch.no_grad():
    for mel, ids in tqdm(test_loader, desc="Generating Submission"):
        mel = mel.to(device)

        outputs = model(mel)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        # Convert ids to CPU numpy / list
        ids = ids.cpu().numpy()

        for sample_id, pred in zip(ids, preds):
            predictions.append((str(sample_id).zfill(4), genres[pred]))

In [ ]:
submission = pd.DataFrame(predictions, columns=["id", "genre"])

submission.to_csv("submission.csv", index=False)

print("Submission file created successfully!")
submission.head()